In [11]:
import ollama

In [69]:
[print(i) for i in ollama.list().models]

model='qwen3.5:9b-mlx' modified_at=datetime.datetime(2026, 11, 22, 11, 6, 5, 802650, tzinfo=TzInfo(10800)) digest='203e30078279db51132b9e026ceb7bb21330e5b1af67ef190671b375c9770404' size=8903014758 details=ModelDetails(parent_model='', format='safetensors', family='', families=None, parameter_size='', quantization_level='nvfp4')
model='qwen3:8b' modified_at=datetime.datetime(2026, 11, 22, 0, 41, 12, 626064, tzinfo=TzInfo(10800)) digest='500a1f067a9f782620b40bee6f7b0c89e17ae61f686b92c24933e4ca4b2b8b41' size=5225388164 details=ModelDetails(parent_model='', format='gguf', family='qwen3', families=['qwen3'], parameter_size='8.2B', quantization_level='Q4_K_M')
model='llama3.2:3b' modified_at=datetime.datetime(2025, 11, 26, 16, 45, 2, 142600, tzinfo=TzInfo(10800)) digest='a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72' size=2019393189 details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='3.2B', quantization_level='Q4_K_M')


[None, None, None]

In [54]:
r_qwen = ollama.chat(model="qwen3:8b", messages=[{"role":"user",
                                        "content":"Истинно ли утверждение: 'Платон был учеником Аристотеля'?"}])

In [48]:
r = ollama.chat(model="llama3.2:3b", messages=[{"role":"user",
                                        "content":"Истинно ли утверждение: 'Платон был учеником Аристотеля'?"}])

In [108]:
r_qwen_mlx = ollama.chat(model="qwen3.5:9b-mlx", think=False, messages=[{"role":"user",
                                        "content":"Истинно ли утверждение: 'Платон был учеником Аристотеля'?"}])

In [118]:
r_qwen_mlx_4b = ollama.chat(model="qwen3.5:4b-mlx", think=False, messages=[{"role":"user",
                                        "content":"Истинно ли утверждение: 'Платон был учеником Аристотеля'?"}])

In [119]:
print('Llama: \n', r.message.content,
      '\n \n QWEN: \n', r_qwen.message.content,
      '\n \n QWEN 3.5 MLX: \n', r_qwen_mlx.message.content,
      '\n \n QWEN 3.5 4B MLX: \n', r_qwen_mlx_4b.message.content
     )

Llama: 
 Ответ: Нет.

Платон был учеником Сократа, а не Аристотеля. Сократ был философом и руководителем Платона, и после смерти Сократа Платон создал Академию в Афинских Воротах, где он собрал группу учеников, в том числе Аристотеля, которого он принял в свои линии. Однако Платон и Аристотель были двумя разными философами с разными идеологиями и методами.

Платон был основателем платонизма, а Аристотель был основателем аристотелизма. Хотя оба философа были влиятельными в античной философии, их идеи и учения былиvery различны.

Напротив, Аристотель был учеником Платона, а не Сократа, и обучался в Академии Платона. После ухода Платона из Академии Аристотель основал свою próprioу академию в Питании, где он teachе был many years. 
 
 QWEN: 
 Утверждение **"Платон был учеником Аристотеля"** **ложное**. 

Правильный порядок:  
**Сократ → Платон → Аристотель**.  

Платон был учеником Сократа, а Аристотель — учеником Платона. Таким образом, Аристотель был учеником Платона, а не наоборот. 
 
 

In [120]:
for response in [r, r_qwen, r_qwen_mlx, r_qwen_mlx_4b]:
    print("\n", response.model)
    print("Причина остановки:", response.done_reason)
    print("Входные токены:", response.prompt_eval_count)
    print("Выходные токены:", response.eval_count)
    print("Время генерации:", response.eval_duration / 1e9)
    print("Токенов/с:", response.eval_count * 1e9 / response.eval_duration)
    # print("Thinking:", len(response.message.thinking))
    # print("Content:", len(response.message.content))


 llama3.2:3b
Причина остановки: stop
Входные токены: 45
Выходные токены: 255
Время генерации: 2.715413
Токенов/с: 93.90836679355958

 qwen3:8b
Причина остановки: stop
Входные токены: 33
Выходные токены: 341
Время генерации: 7.917731
Токенов/с: 43.067894072178

 qwen3.5:9b-mlx
Причина остановки: stop
Входные токены: 32
Выходные токены: 210
Время генерации: 5.500103209
Токенов/с: 38.18110170303169

 qwen3.5:4b-mlx
Причина остановки: stop
Входные токены: 32
Выходные токены: 190
Время генерации: 2.963365916
Токенов/с: 64.11628040065506


In [99]:
import requests

data = requests.post(
    "http://localhost:11434/api/show",
    json={"model": "qwen3.5:9b-mlx"}
).json()

print(data.get("thinking"))

None


In [ ]:
from ollama import chat
from pydantic import BaseModel, Field

use_model = "qwen3.5:4b-mlx"

class Triple(BaseModel):
    subject: str = Field(
    )
    predicate: str = Field(
    )
    object: str = Field(
    )


class Result(BaseModel):
    triples: list[Triple]


response = chat(
    model = use_model,
    think=False,
    messages=[
        {
            "role":"system",
            "content": (
                "Ты извлекаешь отношения из текста в виде триплетов. "
                "Извлекай только явно выраженные отношения. "
                "Не добавляй информацию от себя. "
                "Не повторяй одинаковые триплеты."
            )
        },
        {
            "role": "user",
            "content": (
                "Платон основал Академию в Афинах.\n"
                "Платон был борцом.\n"
                "Платон – ученик Сократа. \n"
                "Платон – борец и ученик Сократа. \n"
                "Ученик Сократа основал Академию в Афинах. \n"
            )
        }
    ],
    format=Result.model_json_schema(),
    options={"temperature": 0},
)

In [ ]:
result = Result.model_validate_json(response.message.content)

for triple in result.triples:
    print(triple)